# 🤖 Chatbot API Batch Testing Tutorial

Welcome to this training notebook! In this tutorial, we will build an automated script to test a Chatbot API. We will learn how to:
1. Authenticate with a backend API (Login/Logout).
2. Send prompts to a chatbot and receive responses.
3. Track performance metrics like latency.
4. Process a batch of test cases using a Pandas DataFrame (Excel in, Excel out).

Let's get started by importing the necessary libraries.

In [ ]:
import time
import pandas as pd
import requests

print("Libraries imported successfully!")

## Step 1: Configuration & Global Variables
It is best practice to keep your configuration variables at the top of your notebook. This makes it easy to change environments, credentials, or file names without digging through the code.

*Note: Be sure to fill in your `USERNAME` and `PASSWORD` before running this cell.*

In [1]:
BASE_URL = "https://chatbot-app.hackathon.buzzperformancecloud.com"
 
# Credentials
USERNAME = "your_username_here"
PASSWORD = "your_password_here"
 
# File configuration
INPUT_EXCEL = "input.xlsx"
OUTPUT_EXCEL = "output.xlsx"
 
# Excel Column Mapping
INPUT_COLUMN = "input_column"
OUTPUT_COLUMN = "output_column"
LATENCY_COLUMN = "latency_seconds"
STATUS_COLUMN = "status"
THREAD_ID_COLUMN = "thread_id"
 
# Timeout to prevent hanging requests
REQUEST_TIMEOUT = 60

## Step 2: The Login Function
To interact with the API, we first need to authenticate. This function sends our credentials to the `/auth/login` endpoint and retrieves a `session_token`. We will use this token in later requests.

In [8]:
def login():
    """Authenticates the user and returns a session token."""
    url = BASE_URL + "/auth/login"
    
    r = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json={
            "username": USERNAME,
            "password": PASSWORD
        },
        timeout=REQUEST_TIMEOUT
    )
 
    if r.status_code != 200:
        raise Exception("Login failed: {} - {}".format(r.status_code, r.text))
 
    return r.json()["session_token"]

## Step 3: The Chat Function
Now that we have a token, we can pass it into the `Authorization` header as a `Bearer` token. This function sends a single message to the chatbot and returns the AI's response and the thread ID.

In [9]:
def chat(token, message):
    """Sends a message to the chatbot and retrieves the response."""
    url = BASE_URL + "/chat"
    
    r = requests.post(
        url,
        headers={
            "Authorization": "Bearer " + token,
            "Content-Type": "application/json"
        },
        json={
            "message": message,
            "thread_id": None
        },
        timeout=REQUEST_TIMEOUT
    )
 
    if r.status_code != 200:
        raise Exception("Chat failed: {} - {}".format(r.status_code, r.text))
 
    data = r.json()
    return data["response"], data["thread_id"]

## Step 4: The Logout Function
It is good security practice to close our session once we are done making requests. This prevents dangling active tokens on the server.

In [10]:
def logout(token):
    """Logs the user out to invalidate the session token."""
    url = BASE_URL + "/auth/logout"
    
    requests.post(
        url,
        headers={"Authorization": "Bearer " + token},
        timeout=REQUEST_TIMEOUT
    )

## Step 5: Wrap the API Calls into a Single Test
We want to measure how long the whole process takes (Latency) and handle any potential errors gracefully without crashing our entire batch run. We wrap `login`, `chat`, and `logout` inside a `try-except` block.

In [11]:
def run_one_test(prompt):
    """Executes a full lifecycle test for a single prompt and measures latency."""
    start = time.time()
    token = None
 
    try:
        # 1. Authenticate
        token = login()
 
        # 2. Get Response
        response, thread_id = chat(token, prompt)
 
        # 3. Cleanup
        logout(token)
 
        # Calculate time taken
        latency = round(time.time() - start, 2)
        return response, thread_id, latency, "success"
 
    except Exception as e:
        # If anything fails, capture the error and latency
        latency = round(time.time() - start, 2)
        return "", "", latency, str(e)

## Step 6: Batch Processing (Main Loop)
Here is where we bring it all together using `pandas`. We will:
1. Read prompts from our Input Excel file.
2. Iterate through every row.
3. Pass the prompt to our `run_one_test()` function.
4. Save the responses, thread IDs, latency, and status back into a new Excel file.

In [13]:
def main():
    print("Loading data from " + INPUT_EXCEL + "...")
    df = pd.read_excel(INPUT_EXCEL)
 
    responses = []
    thread_ids = []
    latencies = []
    statuses = []
 
    total = len(df)
 
    for i, prompt in enumerate(df[INPUT_COLUMN], start=1):
        print("Running test {} / {} ...".format(i, total))
 
        # Handle empty rows
        if pd.isna(prompt):
            responses.append("")
            thread_ids.append("")
            latencies.append(None)
            statuses.append("empty prompt")
            continue
 
        # Execute test
        response, thread_id, latency, status = run_one_test(str(prompt))
 
        # Store results
        responses.append(response)
        thread_ids.append(thread_id)
        latencies.append(latency)
        statuses.append(status)
 
    # Add new columns to our dataframe
    df[OUTPUT_COLUMN] = responses
    df[THREAD_ID_COLUMN] = thread_ids
    df[LATENCY_COLUMN] = latencies
    df[STATUS_COLUMN] = statuses
 
    # Export results
    df.to_excel(OUTPUT_EXCEL, index=False)
    
    print("\n✅ Batch run finished successfully!")
    print("📁 Results saved to: " + OUTPUT_EXCEL)

## Step 7: Execution
Run the cell below to trigger the entire pipeline. Make sure you have your `input.xlsx` file in the same folder as this notebook!

In [ ]:
# Trigger the batch test
if __name__ == "__main__":
    main()